![mental_health_banner_1774025040503.jpg](../mental_health_banner_1774025040503.jpg "mental_health_banner_1774025040503.jpg")

### Carga de Datos 

Notebook de carga de Datos

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalogo", "health_db_catalog")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "gold")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_quality_sleep = spark.table(f"{catalogo}.{esquema_source}.quality_sleep")
df_lifestyle_habits = spark.table(f"{catalogo}.{esquema_source}.lifestyle_habits")
df_nutrition = spark.table(f"{catalogo}.{esquema_source}.nutrition")
df_temperature = spark.table(f"{catalogo}.{esquema_source}.temperature")

In [0]:
df_mental_health =(
    df_quality_sleep.alias("qs").join(df_lifestyle_habits.alias("lh"), on="person_id", how="inner")
    .join(df_nutrition.alias("nut"), on="person_id", how="inner")
    .join(df_temperature.alias("temp"), on="state", how="inner")
    .select(
        col("qs.person_id"),
        col("qs.gender"),
        col("qs.age"),
        col("qs.occupation"),
        col("qs.state"),
        when((col("qs.vitaminD") > 15) & (col("temp.avg_temp") > 50), "High").otherwise("Low").alias("Melatonin"),
        when(col("qs.sleep_disorder") != "None", False).otherwise(True).alias("is_sleep_disorder"),
        when(col("lh.physical_activity_level") > 40, "High").otherwise("Low").alias("physical_activity_level"),
        col("nut.BMI_category").alias("BMI_category")
    )
)

In [0]:
df_mental_health_final = (
    df_mental_health.filter((col("is_sleep_disorder") == True) & (col("BMI_category") == "Overweight") &  (col("physical_activity_level") == "Low") & (col("Melatonin") == "Low"))
    .groupBy(col("state")).agg(
        count(col("person_id")).alias("mental_health_disorders_count")
    )
    .orderBy(col("state"))
)

In [0]:
df_mental_health_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.mental_health_disorders_by_health")